# Classification: predicting credit default

## Introduction

How can you use information from a loan application to estimate the chance of default? This notebook revisits supervised learning through a worked **binary classification** example: a logistic regression (or *logit*) model fitted to synthetic credit data.

By the end, you can visualize a binary outcome, fit a model with scikit-learn, distinguish class predictions from probabilities, reconstruct probabilities from fitted parameters, and read a two-feature decision surface.

**Outline:** [Key Concepts](#key-concepts) · [Credit data and visualization](#credit-data-and-visualization) · [Logistic regression](#logistic-regression) · [Visualizing-logistic-regression](#visualizing-logistic-regression) · [Summary](#summary) · [Tasks](#tasks)

**Lecture connections:** Forecasting & Prediction, slides 4–5 (learning from labeled applications) and 29–30 (held-out data); Classification, slides 8, 14–25 and 29 (credit risk, logistic probabilities and class decisions).

We'll use **pandas** to explore the credit data, **NumPy** to reconstruct probabilities, and **Matplotlib** to visualize the results. **Scikit-learn** provides the data split and logistic regression model. The imports are collected below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

## Key Concepts

For each of $n$ borrowers, you observe two application-time features ($d=2$): bureau score and debt/income. The feature vector $\mathbf{x}_i$ contains these values; the rows of $\mathbf{X}$ correspond to borrowers. The observed outcome is $y_i=1$ for default within the following 12 months and $y_i=0$ for no default.

Logistic regression first computes a **linear predictor** and then maps it to a probability:

$$\widehat{\eta}_i=\widehat{\beta}_0+\widehat{\beta}_1x_{i1}+\widehat{\beta}_2x_{i2},
\qquad
\widehat{p}_i=\frac{1}{1+\exp(-\widehat{\eta}_i)}.$$

Here $x_{i1}$ is bureau score, $x_{i2}$ is debt/income in percent, and the hatted coefficients are estimated from training data. The logistic function keeps $\widehat{p}_i$, the estimated probability of **default**, between zero and one. A probability is not an observed outcome: a borrower with predicted risk 0.2 can still default.

A threshold $c$ converts a probability into a class label:

$$\widehat{y}_i(c)=\mathbb{I}\{\widehat{p}_i\geq c\}.$$

The indicator is one when the condition holds and zero otherwise. The examples distinguish the probability model from the class decision.

## Credit data and visualization

### Load the applications

![Classification lecture slide 8: bureau score and debt/income are application features; default is the later outcome.](../resources/2_credit_context.png)

Our synthetic dataset keeps just two features from this application:

| Column | Meaning and units |
|---|---|
| `bureau_score` | Stylized bureau score on a 300–850 point scale; higher means stronger creditworthiness |
| `debt_income` | Monthly debt payments / gross monthly income × 100; 35 means 35%, not 0.35 |
| `default` | 1 = default within 12 months; 0 = no default within 12 months |

These are fictional borrowers. The features are recorded before the outcome, so they could be available when a new application arrives and facilitate predicting repayment.

The CSV is supplied in `data/`; the notebook does not generate it. Paths below are relative to this notebook's folder.

In [ ]:
credit = pd.read_csv('../data/credit_data.csv')
print(f"Shape of the dataset: {credit.shape}")
credit.head()


In [ ]:
credit.describe().round(2)

Selecting the feature columns with a list inside `[...]` produces a two-column **DataFrame**. Selecting the target with one column name produces a one-dimensional **Series**. Both retain the same row index, keeping each application aligned with its outcome.

The next cell reserves 25% of applications for testing before fitting. `stratify=y` approximately preserves the class proportions, and `random_state` makes the split reproducible. We use the training applications for exploratory plots and keep the test cases for later predictions.

In [ ]:
feature_names = ['bureau_score', 'debt_income']
X = credit[feature_names]
y = credit['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(f'Training applications: {len(X_train)}; test applications: {len(X_test)}')
print(f'Training default share: {y_train.mean():.1%}')

### Binary target versus one feature

The first scatter plot places bureau score on the horizontal axis and observed default on the vertical axis. `alpha` makes points translucent so overlapping observations are easier to see. We keep the actual binary values rather than adding vertical noise.

Before running the cell, anticipate the vertical positions at which points can appear.

In [ ]:
# Create a scatter plot of the default indicator against the bureau score, colored by default status
class_colors = ['#2166ac', '#b2182b']
class_labels = ['No default (0)', 'Default (1)']
fig, ax = plt.subplots(figsize=(8, 3.8))
for label in [0, 1]:
    selected = y_train == label
    ax.scatter(X_train.loc[selected, 'bureau_score'], y_train.loc[selected],
               color=class_colors[label], alpha=0.25, s=24, label=class_labels[label])
ax.set(xlabel='Bureau score (points)', ylabel='Observed default',
       yticks=[0, 1], ylim=(-0.15, 1.15), title='A binary target creates two horizontal bands')
ax.legend()
plt.tight_layout()
plt.show()

The two bands are a consequence of the binary target, not a plotting error. Compare the score ranges within the bands: defaulting borrowers tend to have lower scores, but the bands overlap substantially. A single score cannot tell you the future outcome with certainty.

Task 1 returns to this plot with `debt_income` on the horizontal axis.

### Two features, colored by outcome

The next plot uses both axes for features and shows the outcome through color. This is the feature-space view from the lecture. Look for regions where defaults are more common, rather than expecting perfectly separated clusters.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label in [0, 1]:
    selected = y_train == label
    ax.scatter(X_train.loc[selected, 'bureau_score'],
               X_train.loc[selected, 'debt_income'],
               color=class_colors[label], alpha=0.45, s=26, label=class_labels[label])
ax.set(xlabel='Bureau score (points)', ylabel='Debt / income (%)',
       title='Credit applications in two-feature space')
ax.legend()
plt.tight_layout()
plt.show()

Lower bureau scores and higher debt/income tend to coincide with defaults in these data. The mixed colors also show why a probabilistic model is appropriate. These predictive associations do not establish causal effects.

## Logistic regression

### Fit the model

**Scikit-learn** estimators share a common interface. In the code below, model creation is followed by `.fit(X_train, y_train)`, which estimates its parameters. Prediction methods then apply the fitted model to new feature rows. Logistic regression learns its parameters using a probability-based objective, rather than fitting a line directly to the zeros and ones.

We use the `newton-cholesky` solver, which works well for this small, two-feature problem and lets us keep the original feature units. Other settings, including scikit-learn's default regularization, stay at their defaults.

In [ ]:
model = LogisticRegression(solver='newton-cholesky')
model.fit(X_train, y_train)

Attributes ending in `_` become available after fitting. `intercept_` contains the fitted intercept; `coef_` contains the coefficients in the order of the training feature columns. `classes_` tells you the order of class labels used by the prediction methods.

In [ ]:
parameters = pd.Series(
    [model.intercept_[0], *model.coef_[0]],
    index=['intercept', *feature_names],
    name='Estimated parameter'
)
print(parameters.round(5))
print('Class order:', model.classes_)

### `.predict` versus `.predict_proba`

The example selects five held-out applications using `.iloc[:5]` (the first five rows by position). `.predict` returns one class label per row. `.predict_proba` returns one probability per class: here column 0 is no default, and column 1 is default, as confirmed by `classes_`.

These applications retain the feature names, order and units used in training.

In [ ]:
applications = X_test.iloc[:5]
class_predictions = model.predict(applications)
class_probabilities = model.predict_proba(applications)

prediction_table = applications.copy()
prediction_table['P(no default)'] = class_probabilities[:, 0]
prediction_table['P(default)'] = class_probabilities[:, 1]
prediction_table['predicted_class'] = class_predictions
prediction_table['observed_default'] = y_test.loc[applications.index]
prediction_table.round(3)

Each row's two probabilities sum to one. The class label discards how close the default probability is to the decision threshold; inspect the probabilities to see the model's degree of confidence. The observed outcome is shown for comparison, but it was not supplied to either prediction method.

### Compute probabilities from the estimated parameters

The next cell reconstructs the same five predictions without calling `.predict_proba`: it multiplies each feature by its estimated coefficient, adds the intercept, and applies the logistic function. This is the numerical equivalent of the formula in Key Concepts.

The first row below prints every term so you can check the calculation with a calculator. The remaining rows use exactly the same calculation.

In [ ]:
intercept = model.intercept_[0]
bureau_coefficient, debt_coefficient = model.coef_[0]
linear_predictor = (
    intercept
    + bureau_coefficient * applications['bureau_score']
    + debt_coefficient * applications['debt_income']
)
manual_probability = 1 / (1 + np.exp(-linear_predictor))

first = applications.iloc[0]
print(f'First application: eta = {intercept:.6f}'
      f' + ({bureau_coefficient:.6f}) * {first["bureau_score"]:.0f}'
      f' + ({debt_coefficient:.6f}) * {first["debt_income"]:.1f}'
      f' = {linear_predictor.iloc[0]:.6f}')
print(f'P(default) = 1 / (1 + exp(-eta)) = {manual_probability.iloc[0]:.6f}')
pd.DataFrame({
    'linear_predictor': linear_predictor,
    'manual_probability': manual_probability,
    'sklearn_probability': class_probabilities[:, 1]
}).round(6)

The manually reconstructed probabilities agree with scikit-learn's output. The calculation uses the full fitted parameters; rounding them first can create discrepancies.

### From probabilities to classes: the fixed threshold

![Classification lecture slide 25: a horizontal cut-off converts a logistic probability curve into class decisions.](../resources/2_classification_threshold.png)

**Read the target carefully:** the slide codes *repayment* as 1, so its curve shows repayment probability. Here we code **default as 1**, so our probabilities have the opposite meaning.

For binary logistic regression, `.predict` chooses the class with the larger probability. With classes `[0, 1]`, this corresponds to default when its probability **exceeds 0.5**. At an exact tie, scikit-learn chooses class 0. Our mathematical rule uses `>=`; the difference matters only at exactly 0.5.

The cell below reconstructs `.predict` using its threshold convention. A threshold of 0.5 is a software default, not an automatically suitable lending policy. Choosing a business threshold requires information about the consequences of decisions.

In [ ]:
threshold_classes = (class_probabilities[:, 1] > 0.5).astype(int)
pd.DataFrame({
    'P(default)': class_probabilities[:, 1],
    'predict': class_predictions,
    'threshold_0.5': threshold_classes
}, index=applications.index).round(3)

As a brief held-out check, `.score` reports classification accuracy: the fraction of test labels predicted correctly at the default threshold. Compare it with always predicting the majority class identified from training data. Neither number measures the economic value of a lending policy, and neither establishes the quality of every individual probability.

In [ ]:
majority_class = y_train.mode().iloc[0]
print(f'Test accuracy: {model.score(X_test, y_test):.1%}')
print(f'Majority-class test accuracy: {(y_test == majority_class).mean():.1%}')

## Visualizing logistic regression

The final worked example overlays fitted probabilities on the two-feature scatter plot. `np.meshgrid` creates a rectangular grid of bureau scores and debt/income values. The code flattens the grid coordinates into a DataFrame, predicts at every grid point, and reshapes the probabilities back into a rectangle for plotting.

The background represents **estimated default probability**; the points show **observed outcomes on held-out applications**. The black line marks probability 0.5. It separates the class predictions while the background retains their probability information.

In [ ]:
bureau_values = np.linspace(300, 850, 160)
debt_values = np.linspace(5, 65, 160)
bureau_grid, debt_grid = np.meshgrid(bureau_values, debt_values)
grid = pd.DataFrame({
    'bureau_score': bureau_grid.ravel(),
    'debt_income': debt_grid.ravel()
})
grid_probability = model.predict_proba(grid)[:, 1].reshape(bureau_grid.shape)

fig, ax = plt.subplots(figsize=(9, 5.5))
surface = ax.contourf(bureau_grid, debt_grid, grid_probability,
                      levels=np.linspace(0, 1, 21), cmap='RdBu_r', vmin=0, vmax=1)
fig.colorbar(surface, ax=ax, label='Estimated probability of default')
boundary = ax.contour(bureau_grid, debt_grid, grid_probability,
                      levels=[0.5], colors='black', linewidths=1.5)
ax.clabel(boundary, fmt={0.5: 'P(default) = 0.5'}, fontsize=9)
for label in [0, 1]:
    selected = y_test == label
    ax.scatter(X_test.loc[selected, 'bureau_score'], X_test.loc[selected, 'debt_income'],
               color=class_colors[label], edgecolor='white', linewidth=0.5,
               s=32, alpha=0.85, label=class_labels[label])
ax.set(xlabel='Bureau score (points)', ylabel='Debt / income (%)',
       title='Logistic regression: probabilities and held-out outcomes')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

Follow the background from high bureau scores and low debt/income toward low scores and high debt/income: predicted default risk rises. Defaults can still occur in lower-risk regions, and non-defaults in higher-risk regions.

Why is the 0.5 boundary straight even though the logistic function is curved? Probability 0.5 corresponds to a linear predictor of zero. With two features and no interaction terms, that equation describes a straight line. The full rectangular background includes sparse combinations near the corners; predictions in those regions have less support from the data.

## Summary

- A binary scatter plot puts observations on two bands; a feature-space scatter plot uses color to show the target.
- `.fit(X_train, y_train)` estimates parameters; predictions depend on retaining the training feature order and units.
- `.predict_proba(X)[:, 1]` gives default probabilities here because `model.classes_` is `[0, 1]`; `.predict(X)` gives labels using the default decision threshold.
- Applying `1 / (1 + np.exp(-linear_predictor))` to the intercept plus weighted features reconstructs a fitted probability.
- A probability surface shows gradual changes in risk; its decision boundary shows where class predictions switch.

Here's a quick reference for the new Python tools introduced in this notebook:

| Python pattern | What it helps you do |
|---|---|
| `.iloc` | Select borrowers by row position |
| `LogisticRegression` | Fit a binary probability model |
| `model.classes_`, `model.predict_proba(X)` | Identify class order and obtain class probabilities |
| `np.exp` | Apply the logistic function to a linear predictor |
| `model.score(X_test, y_test)` | Calculate classification accuracy |
| `np.linspace`, `np.meshgrid`, `.ravel()`, `.reshape()` | Build and arrange a grid of feature combinations |
| `ax.scatter`, `ax.contourf`, `ax.contour`, `fig.colorbar` | Plot outcomes, probability surfaces and decision boundaries |

## Tasks

Use the working cells below. Each task builds on a pattern you have already used. Write a short interpretation alongside your code.

### Task 1: Change the single-feature view

Plot observed default against **debt/income** for the training applications. Label the horizontal axis in percent and retain the binary vertical axis. Where do defaults appear more frequently, and what does the overlap tell you?

*Hint:* adapt the first scatter plot and keep the target unchanged.

In [ ]:
# YOUR CODE: plot default against debt/income.

### Task 2: Inspect an application

Choose a different row from `X_test`. Reconstruct its default probability from the fitted parameters, compare it with `.predict_proba`, and report its `.predict` label. Explain why the probability and label answer different questions.

*Hint:* `X_test.iloc[[10]]` keeps a one-row DataFrame suitable for scikit-learn.

In [ ]:
# YOUR CODE: reconstruct and compare one held-out prediction.

### Task 3: Read the fitted map

Add contours at probabilities 0.25 and 0.75 to the final plot, using the already computed grid probabilities. Label the contours and identify the region between them. What information does this view retain that a plot of predicted classes alone would hide?

*Hint:* use `ax.contour(..., levels=[0.25, 0.75])` while rebuilding the figure.

In [ ]:
# YOUR CODE: extend the probability-surface visualization.